In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from difflib import SequenceMatcher
from tqdm.auto import tqdm

## Add volcano name as a field
Cross reference against GVP holocene and pleistocene volcano lists using fuzzy name matching as a priority. 
Alternatively, find particular vlaues in location field that are clearly volano names. 
Otherwise define volcano as np.nan

In [7]:
holocene = pd.read_csv('DATA/GVP_HOLOCENE_VOLCS.csv')
pleistocene = pd.read_csv('DATA/GVP_PLEISTOCENE_VOLCS.csv')
VOLCANO_DATABASE = pd.concat([holocene,pleistocene], ignore_index=True)

df = pd.read_csv('DATA/R_arcs - DO NOT EDIT.csv', low_memory=False)

In [6]:
def Similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

def match_names(location, volcano_list):
    location_features = location.split(' / ')
    candidates = {}
    for i in location_features:
        for j in volcano_list: 
            similarity = Similar(j, i)
            if similarity > 0.9:
                candidates[j] = similarity
    
    if len(candidates) == 0:
        VOLC_COMPLEX = [item for item in location_features if 'VOLCANIC COMPLEX' in item]
        VOLC_FIELD = [item for item in location_features if 'VOLCANIC FIELD' in item]
        VOLC_CENTER = [item for item in location_features if 'VOLCANIC CENTER' in item]
        if len(VOLC_COMPLEX) != 0:
            volcano = VOLC_COMPLEX[0]
        elif len(VOLC_FIELD) != 0:
            volcano = VOLC_FIELD[0]
        elif len(VOLC_CENTER) != 0:
            volcano = VOLC_CENTER[0]
        else:
            volcano = np.nan
            
    elif len(candidates) > 1:
        volcano = max(candidates, key=candidates.get)
    else:
        volcano = list(candidates.keys())[0]
    
    return volcano
        
locations = list(df['LOCATION'].unique())

def switch_suffixes(string):
    if ',' in string:
        pieces = string.split(', ')
        new_string = f'{pieces[1]} {pieces[0]}'
    else:
        new_string = string
    return new_string

volcano_list = VOLCANO_DATABASE['Volcano Name'].to_list()
volcano_list = [i.upper() for i in volcano_list]
volcano_list = [switch_suffixes(i) for i in volcano_list]

match_dict = {}
for location in tqdm(locations):
    match_dict[location] = match_names(location, volcano_list)
    
def assign_volcano(location):
    volcano_name = match_dict[location]
    return volcano_name

df['VOLCANO'] = df.apply(lambda x: assign_volcano(x['LOCATION']), axis=1)

df.to_csv('DATA/R_arcs_edited.csv')

  0%|          | 0/7433 [00:00<?, ?it/s]